**Copyright 2026 The MediaPipe Authors. All Rights Reserved.**

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Embedding with MediaPipe Tasks

This notebook shows you how to use the MediaPipe Tasks Python API (`UniversalEmbedder`) to extract embeddings from text and image inputs into a shared semantic vector space and compare their similarity using cosine similarity. These values range from -1 to 1, with 1 indicating identical semantic meaning.

## Preparation

Let's start with installing MediaPipe.

In [ ]:
!pip install -q mediapipe

After installing your dependencies, download an EmbeddingGemma 2 (LiteRT-LM) universal embedder model for this example:

* **Text (270M)**: `https://huggingface.co/litert-community/embeddinggemma-2-text-270m-litert-lm/resolve/main/embeddinggemma-2-text-270m.litertlm`
* **Text + Vision (440M)**: `https://huggingface.co/litert-community/embeddinggemma-2-text-vision-440m-litert-lm/resolve/main/embeddinggemma-2-text-vision-440m.litertlm`
* **Multimodal (740M)**: `https://huggingface.co/litert-community/embeddinggemma-2-740m-litert-lm/resolve/main/embeddinggemma-2-740m.litertlm`

Below we download the Text + Vision (440M) model (`embeddinggemma-2-text-vision-440m.litertlm`).

In [ ]:
#@title Start downloading here.
!wget -O embeddinggemma-2-text-vision-440m.litertlm -q https://huggingface.co/litert-community/embeddinggemma-2-text-vision-440m-litert-lm/resolve/main/embeddinggemma-2-text-vision-440m.litertlm

Next, download two sample images (`burger.jpg` and `cat.jpg`) to test image and cross-modal embedding.

In [ ]:
import urllib.request

IMAGE_FILENAMES = ['burger.jpg', 'cat.jpg']

for name in IMAGE_FILENAMES:
  url = f'https://storage.googleapis.com/mediapipe-tasks/image_classifier/{name}'
  urllib.request.urlretrieve(url, name)

## Running text embedding inference

To run inference using the MediaPipe `UniversalEmbedder` task, initialize the `UniversalEmbedder` using the model that you downloaded earlier, and then call `embed_text` to compare two separate text strings. You can edit `first_text` and `second_text` in the parameters below.

In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python.components.utils import cosine_similarity
from mediapipe.tasks.python.retrieval import universal_embedder

# Create your base options with the model that was downloaded earlier
base_options = python.BaseOptions(
    model_asset_path='embeddinggemma-2-text-vision-440m.litertlm'
)

# Set your value for L2 normalization
l2_normalize = True  #@param {type:"boolean"}

# Create the final set of options for the UniversalEmbedder
options = universal_embedder.UniversalEmbedderOptions(
    base_options=base_options,
    l2_normalize=l2_normalize,
)

with universal_embedder.UniversalEmbedder.create_from_options(
    options
) as embedder:
  first_text = 'A delicious cheeseburger with lettuce and tomato'  #@param {type:"string"}
  second_text = 'A fast food hamburger on a sesame bun'  #@param {type:"string"}

  # Convert both strings to embeddings
  first_embedding_result = embedder.embed_text(first_text)
  second_embedding_result = embedder.embed_text(second_text)

  # Calculate and print cosine similarity
  similarity = cosine_similarity.cosine_similarity(
      first_embedding_result.embeddings[0],
      second_embedding_result.embeddings[0],
  )
  print(f'Text-to-Text cosine similarity: {similarity:.4f}')

## Cross-modal (Text-to-Image) and Image-to-Image similarity

Because `UniversalEmbedder` projects text and images into the same embedding space, you can pass encoded image bytes to `embed_image` and directly compare text queries against images using cosine similarity.

In [ ]:
with open('burger.jpg', 'rb') as f:
  burger_bytes = f.read()
with open('cat.jpg', 'rb') as f:
  cat_bytes = f.read()

with universal_embedder.UniversalEmbedder.create_from_options(
    options
) as embedder:
  query_text = 'a photo of a cute cat'  #@param {type:"string"}

  # Extract text and image embeddings
  query_result = embedder.embed_text(query_text)
  burger_result = embedder.embed_image(burger_bytes)
  cat_result = embedder.embed_image(cat_bytes)

  # Compute cross-modal and image-to-image cosine similarity
  query_to_cat = cosine_similarity.cosine_similarity(
      query_result.embeddings[0], cat_result.embeddings[0]
  )
  query_to_burger = cosine_similarity.cosine_similarity(
      query_result.embeddings[0], burger_result.embeddings[0]
  )
  image_to_image = cosine_similarity.cosine_similarity(
      cat_result.embeddings[0], burger_result.embeddings[0]
  )

  print(f'"{query_text}" vs. cat.jpg:    {query_to_cat:.4f}')
  print(f'"{query_text}" vs. burger.jpg: {query_to_burger:.4f}')
  print(f'cat.jpg vs. burger.jpg:        {image_to_image:.4f}')